# Unit 5, Lecture 4: Serving the agent behind an API

Your agent works, it is tested, validated, and observable. But it still runs in
your notebook, where only you can reach it. To be a **product**, it must become a
**service** anyone can call over the network. That is an **API**.

An API is a **contract**: send me a request shaped like this, and I will send you
a response shaped like that. The agent becomes a function behind an address (an
**endpoint**).

Built on **FastAPI**, and the whole API is testable **offline** with a
`TestClient`, no running server needed.

## The contract: write the shapes down

A caller must send `text`. The service promises `team` and `escalate`. Declaring
these types **is** the contract, and the framework generates validation, errors,
and docs from them.

In [ ]:
from cse476.serving import TicketRequest, TriageResponse

print("request contract:", TicketRequest.model_fields)
print("response contract:", TriageResponse.model_fields)

## The agent did not change

`triage` is the same plain, testable function you have written since Unit 1.
Serving wraps it; it does not touch it.

In [ ]:
from cse476.serving import triage

# still a plain function, testable on its own:
print(triage("I want a refund, this is urgent"))
print(triage("the app has a bug"))
print(triage("just a question"))

## Test the whole API offline, no server

A `TestClient` calls your endpoints in-process, so the API is testable exactly
like the rest of your code: fast, offline, deterministic.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from fastapi.testclient import TestClient
from cse476.serving import make_app

client = TestClient(make_app())

# the health endpoint (a heartbeat for load balancers)
print("health:", client.get("/health").json())

# the real work
r = client.post("/triage", json={"text": "refund me, this is urgent"})
print("triage:", r.status_code, r.json())

## The contract catches bad requests, for free

A request missing `text` never reaches the agent. The contract rejects it with a
422, automatically, because you declared the shape. This is input validation you
did not write.

In [ ]:
# a malformed request (no "text" field):
bad = client.post("/triage", json={"wrong_field": 123})
print("malformed request ->", bad.status_code, "(422 = rejected by the contract)")
print()

# an empty body, also rejected:
empty = client.post("/triage", json={})
print("empty body ->", empty.status_code)
print()
print("-> the agent only ever runs on well-formed input. A whole class of bugs, gone.")

This mirrors Lecture 2: there you validated the **model's output**; here the
contract validates the **caller's input**. Both are gates, at different doors.

## The mapping, and why an API

In [ ]:
from cse476.serving import SERVING_MAP, why_an_api

for concept, meaning in SERVING_MAP.items():
    print(f"{concept:24} ->  {meaning}")
print()
for k, v in why_an_api().items():
    print(f"{k:14}: {v}")

## Your turn

**1. Serve your agent.** Wrap one capstone agent in a `make_app` with a request
model, a response model, a `/triage`-style endpoint, and a `/health` endpoint.

**2. Test it offline.** Write three `TestClient` tests: the health check works, a
good request returns the right shape, and a malformed request gets a 422.

**3. Design your contract.** Write down, in a few typed fields, exactly what your
service takes in and returns. That contract is the promise everyone builds
against, so make it clear and small.

In [ ]:
# your work here
